## 주문(orders) - orders, order_items


In [ ]:
## order_items

import os
from supabase import create_client
from datetime import datetime, timezone

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_ANON_KEY")

supabase = create_client(url, key)


class OrderItem_Service:
    def __init__(self):
        data = supabase.auth.get_user()
        if data:
            self.user = data.user
    
    # 주문에 상품 추가하기
    def create_order_item(self, order_id, product_id, item_name, price, quantity):
        order = self.get_order(order_id)
        self.check_authority(user_id=order["user_id"])

        response = (
            supabase.table("order_items")
            .insert({
                "order_id": order_id,
                "product_id": product_id,
                "item_name": item_name,
                "price": price,
                "quantity": quantity
            }).execute()
        )
        return response.data
    
    # 담긴 상품 수량/가격 등 수정
    def update_order_item(self, id, data):
        item = self.get_order_item(id)
        order = self.get_order(item["order_id"])
        self.check_authority(user_id=order["user_id"])

        response = (
            supabase.table("order_items")
            .update(data)
            .eq("id", id)
            .execute()
        )
        return response.data

    # 담긴 상품 삭제하기
    def delete_order_item(self, id):
        item = self.get_order_item(id)
        order = self.get_order(item["order_id"])
        self.check_authority(user_id=order["user_id"])

        response = (
            supabase.table("order_items")
            .update({"deleted_at": datetime.now(timezone.utc).isoformat()})
            .eq("id", id)
            .execute()
        )
        return response.data

    # 주문 정보/상품 정보 조회하기
    def get_order_item(self, id):
        response = (
            supabase.table("order_items")
            .select("*")
            .eq("id", id)
            .is_("deleted_at", "null")
            .single()
            .execute()
        )
        return response.data

    # 권한 검사하기
    def get_order(self, order_id):
        response = (
            supabase.table("orders")
            .select("*")
            .eq("id", order_id)
            .single()
            .execute()
        )
        return response.data
    
    def check_authority(self, user_id):
        if not self.user or user_id != self.user.id:
            raise PermissionError("본인 주문의 상품만 처리할 수 있습니다.")